# Stage 1: MiniMind 白盒拆解與 Pre-training 實戰

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EndeavorYen/LLM-learning/blob/main/notebooks/01_stage1_minimind/01_minimind_pretrain_colab.ipynb)

本 Notebook 帶領你完成第一階段（Stage 1）的核心目標：
1. **環境檢查與 Google Drive 綁定**（防止斷線中斷丟失權重）
2. **下載 MiniMind 專案與輕量語料**
3. **張量維度深入拆解**：剖析 `input_ids`、`logits`、Causal Mask 與 `CrossEntropyLoss` 的底層張量流動
4. **SFT Prompt Loss Masking 原理**：親自理解標籤 `-100` 的作用
5. **啟動 Pre-training 訓練迴圈** 並觀察 Loss 收斂
6. **推論測試**（從隨機亂碼到具備基本字詞流暢度）

## 1. 環境檢查 (GPU & CUDA)

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Device Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. 掛載 Google Drive (保存 Checkpoint)
> **注意**：Colab 的本地硬碟在 session 斷開後會被完全清空。將訓練 Checkpoint 存於 Google Drive 是必備操作。

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_save_dir = '/content/drive/MyDrive/LLM_learning_checkpoints/stage1_minimind'
    os.makedirs(drive_save_dir, exist_ok=True)
    print(f"Checkpoints will be backed up to: {drive_save_dir}")
except Exception as e:
    print("Running locally or Google Drive not mounted:", e)

## 3. Clone MiniMind 並安裝相依套件

In [ ]:
# 下載 MiniMind 專案
!git clone https://github.com/jingyaogong/minimind.git /content/minimind
%cd /content/minimind
!pip install -q -r requirements.txt

## 4. 深度解析：Pre-training 張量維度變換 (Tensor Shapes)
在啟動訓練前，我們先用一小段原生 PyTorch 代碼看清 Transformer Auto-regressive 的 Loss 計算：

In [ ]:
import torch
import torch.nn as nn

# 模擬 Batch 參數
batch_size = 2
seq_len = 5
vocab_size = 1000

# 模擬模型的輸入與標籤
# 假設文本句子是: [TokenA, TokenB, TokenC, TokenD, TokenE]
input_ids = torch.randint(0, vocab_size, (batch_size, seq_len))

# 自回歸核心：預測下一個 Token
# 輸入取前 seq_len - 1 個 token
# 目標取第 1 到最後一個 token
inputs = input_ids[:, :-1]   # Shape: [batch_size, seq_len - 1] -> [2, 4]
labels = input_ids[:, 1:]    # Shape: [batch_size, seq_len - 1] -> [2, 4]

# 假設經過 Transformer 後輸出的 logits
logits = torch.randn(batch_size, seq_len - 1, vocab_size) # Shape: [2, 4, 1000]

# CrossEntropyLoss 期望的輸入維度: [N, C] 與目標 [N]
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
loss = loss_fn(logits.reshape(-1, vocab_size), labels.reshape(-1))

print(f"Inputs shape: {inputs.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Logits shape: {logits.shape}")
print(f"Flattened logits shape: {logits.reshape(-1, vocab_size).shape}")
print(f"CrossEntropy Loss: {loss.item():.4f}")

## 5. 深度解析：SFT 中的 Loss Masking (-100)
為什麼 SFT 不能計算整個句子的 Loss？
- 如果對整個句子（Prompt + Answer）都算 Loss，模型會浪費參數量在「背誦使用者的提問」。
- PyTorch 的 `CrossEntropyLoss` 預設 `ignore_index=-100`。
- 我們只要將 Prompt 區間的 label 替換為 `-100`，計算梯度時就會完全略過 Prompt，只計算 Answer 的損失！

In [ ]:
# 範例演示：Prompt 區域打上 -100
# 句子: [Prompt_tok1, Prompt_tok2, Answer_tok1, Answer_tok2]
prompt_len = 2
seq = torch.tensor([[101, 205, 308, 409]]) # shape: [1, 4]

input_tokens = seq[:, :-1] # [101, 205, 308]
target_tokens = seq[:, 1:].clone() # [205, 308, 409]

# 將 Prompt 區間在 target 設為 -100
# 第 0 個位置預測 205 (Prompt 結尾)，第 1 個位置預測 308 (Answer 開始)
target_tokens[:, :prompt_len-1] = -100 

print(f"Input tokens:  {input_tokens.tolist()[0]}")
print(f"Target labels: {target_tokens.tolist()[0]}  (-100 代表不計算損失)")

## 6. 執行 MiniMind Pre-training
執行預訓練腳本，觀察訓練時的 Loss 收斂狀況。

In [ ]:
%cd /content/minimind
# 執行預訓練 (單卡訓練，約數分鐘到半小時即可看到收斂趨勢)
# 可透過參數調整 batch_size 與 learning rate
!python train_pretrain.py --epochs 1 --batch_size 32 --learning_rate 5e-4

## 7. 備份權重至 Google Drive

In [ ]:
# 將訓練產出的 out/ 目錄同步到 Google Drive
!cp -r /content/minimind/out/* /content/drive/MyDrive/LLM_learning_checkpoints/stage1_minimind/ 2>/dev/null || echo "Out dir not created or Drive not available"

## 8. 推論測試 (Text Generation)

In [ ]:
# 測試已產生的權重推論
!python eval_model.py --load 0